In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql.functions import *
import os
import datetime

In [ ]:
# 기존 Spark 세션이 남아 있다면 종료
spark = SparkSession.getActiveSession()
if spark:
    spark.stop()

In [ ]:
spark = SparkSession.builder \
    .appName("W5M2_Analysis") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "LEGACY") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "80") \
    .config("spark.sql.files.maxPartitionBytes", "512m") \
    .getOrCreate()

In [ ]:
data_dir = "./data/NYC-TLC"
parquet_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".parquet")]

# 모든 Parquet 파일을 하나의 DataFrame으로 로드
df = spark.read.parquet(*parquet_files)

df = df.drop("airport_fee")

df = df.withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime")))
df = df.withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime")))

df.printSchema()

In [ ]:
df.show(5)

In [ ]:
# 컬럼명 출력 (인덱스와 함께 표시) (데이터 정제 전 컬럼 인덱스 확인 차)
columns = df.columns
print("📌 컬럼 인덱스 매핑:")
for i, colu in enumerate(columns):
    print(f"{i}: {colu}")

In [ ]:
df_no_missing = df.dropna()

In [ ]:
# ✅ 숫자형 컬럼 리스트
numeric_cols = ["passenger_count", "trip_distance", "RatecodeID", "fare_amount",
                "extra", "mta_tax", "tip_amount", "tolls_amount",
                "improvement_surcharge", "total_amount", "congestion_surcharge"]

# ✅ DataFrame 필터링 (dropoff 시간이 pickup 시간보다 빠른 경우 제거 + numeric_cols 음수 제거 + fare_amount > 0 필터링)
df_filtered = df_no_missing.filter(
    (col("tpep_dropoff_datetime") >= col("tpep_pickup_datetime")) &  # 🔹 dropoff 시간이 pickup 시간보다 빠른 경우 제거
    (col("fare_amount") > 0)  # 🔹 fare_amount가 0보다 큰 경우 유지
)

# ✅ numeric_cols에서 음수 값이 있는 행 제거 (컬럼 리스트 활용)
for col_name in numeric_cols:
    df_filtered = df_filtered.filter(col(col_name) >= 0)

# ✅ 연도가 2020년 또는 2021년인 데이터만 유지
df_final = df_filtered.filter(year(col("tpep_pickup_datetime")).isin([2020, 2021]))

# ✅ 캐싱 적용 (반복 연산 방지)
df_final = df_final.cache()

# ✅ 샘플 데이터 확인
df_final.show(5)

In [ ]:
# ✅ Step 1: tpep_pickup_datetime 변환 (날짜만 남김)
df_final = df.withColumn("date", to_date(col("tpep_pickup_datetime")))

# ✅ Step 2: 날짜별 집계 (주행 횟수, 평균 거리, 평균 운임 계산)
df_grouped = df_final.groupBy("date").agg(
    count("*").alias("total_trips"),  # 주행 횟수
    avg("trip_distance").alias("avg_distance"),  # 평균 운행 거리
    avg("fare_amount").alias("avg_fare_amount")  # 평균 운임
)

# ✅ Step 3: 날씨 데이터 로드
weather_schema = "date STRING, temp_max DOUBLE, temp_min DOUBLE, precipitation DOUBLE"
df_weather = spark.read.csv("data/nyc_weather.csv", header=True, schema=weather_schema)

# ✅ Step 4: 날짜 기준으로 Join
df_transformed = df_grouped.join(df_weather, "date", "left")

In [ ]:
# ✅ 제한된 수의 샘플만 수집
sample_data = df_transformed.limit(5).collect()
print("📌 데이터 샘플 (최대 5개):")
for row in sample_data:
    print(row)

In [ ]:
local_W5M2_output_path = "data/W5M2_output.parquet"

# 🔹 로컬 저장 (Jupyter 컨테이너 내부)
df_transformed.write.mode("overwrite").parquet(local_W5M2_output_path)
print(f"✅ 로컬에 저장 완료: {local_W5M2_output_path}")